### Setup

In [3]:
import os

root_dir = "/tools/scratch/vinaya/power-prediction" ##CHANGE TO YOUR REPO's Path
gemmini_dir = f"{root_dir}/power-mappings-chipyard/generators/gemmini/software/gemmini-rocc-tests"
vlsi_dir = f"{root_dir}/power-mappings-chipyard/vlsi"
my_rtl = "CustomGemminiSoCConfig"
bsub = "bsub -q rhel7"

micro_benchmarks = [    
    "mvin_cache_hit_microbenchmark_random",
    "mvout_microbenchmark_random",
    "preload_and_compute_sparse_100",
    "preload_and_compute_sparse_60",
    "preload_and_compute_sparse_20",
    "preload_and_compute_sparse_0",
    "preload_and_compute_random",
    "preload_and_compute_random_test2",
    "simple"]

validity_benchmarks = [
    "tiled_matmul_ws_random_benchmark", 
    "conv_first_layer_benchmark",
    "conv_first_layer_slow_benchmark",
    "mlp_random_benchmark"]

my_workloads = validity_benchmarks[2:3]
validity_test = True
overwrite = True
print(my_workloads)

['conv_first_layer_slow_benchmark']


### Generate Benchmarks

In [4]:
print("Make sure chipyard environment sourced")
print(f"cd {root_dir}/scripts")
if not validity_test:
    print("cp -R templates/bareMetalC/. ../power-mappings-chipyard/generators/gemmini/software/gemmini-rocc-tests/bareMetalC")
    print("cd ../power-mappings-chipyard/generators/gemmini/software/gemmini-rocc-tests")
    print("./build.sh")
else:
    print("cp -R templates/validity_benchmarks/. ../power-mappings-chipyard/generators/gemmini/software/gemmini-rocc-tests/bareMetalC")
    print("cd ../power-mappings-chipyard/generators/gemmini/software/gemmini-rocc-tests")
    print("./build.sh")

print(f"cd ../../../../..")

Make sure chipyard environment sourced
cd /tools/scratch/vinaya/power-prediction/scripts
cp -R templates/validity_benchmarks/. ../power-mappings-chipyard/generators/gemmini/software/gemmini-rocc-tests/bareMetalC
cd ../power-mappings-chipyard/generators/gemmini/software/gemmini-rocc-tests
./build.sh
cd ../../../../..


### Spike

In [5]:
class CommandScript:
    def __init__(self, path):
        self.path = path
        # if cd is not None: print(f"cd {cd}")
        self.f = open(path,'w')
        self.f.write("#!/bin/bash\n")
    def close(self):
        self.f.close()
        os.chmod(self.path, 0o755)
        print(f"./{self.path.split('/')[-1]}")

In [6]:
print("Make sure chipyard environment sourced")
print(f"cd {gemmini_dir}")
script = CommandScript(f"{root_dir}/power-mappings-chipyard/generators/gemmini/software/gemmini-rocc-tests/run-spike.sh")
for w in my_workloads:
    spike_fpath = f"{root_dir}/data/spike_output/{w}-spike_output.log"
    if overwrite or not os.path.exists(spike_fpath):
        script.f.write(f"spike --extension=gemmini --log-commits --log=output/test.out build/bareMetalC/{w}-baremetal &> {spike_fpath}\n")
script.close()

print("cd ../../../../..")

Make sure chipyard environment sourced
cd /tools/scratch/vinaya/power-prediction/power-mappings-chipyard/generators/gemmini/software/gemmini-rocc-tests
./run-spike.sh
cd ../../../../..


### RTL Sim W/ Spike Log Dumping

In [26]:
print("Make sure chipyard environment sourced")
print(f"cd {vlsi_dir}")
# build SoC config
print(f"make buildfile CONFIG={my_rtl}")
# build RTL simulation (but don't run it)
print(f"""make redo-sim-rtl-debug args="--stop_before_step run_simulation" CONFIG={my_rtl} BINARY={gemmini_dir}/build/bareMetalC/{micro_benchmarks[0]}-baremetal""")
# RUN ONCE - to generate SoC config + RTL sim executable

Make sure chipyard environment sourced
cd /tools/scratch/vinaya/power-prediction/power-mappings-chipyard/vlsi
make buildfile CONFIG=CustomGemminiSoCConfig
make redo-sim-rtl-debug args="--stop_before_step run_simulation" CONFIG=CustomGemminiSoCConfig BINARY=/tools/scratch/vinaya/power-prediction/power-mappings-chipyard/generators/gemmini/software/gemmini-rocc-tests/build/bareMetalC/mvin_cache_hit_microbenchmark_random-baremetal


In [8]:
print("Make sure chipyard environment sourced")
print(f"cd {vlsi_dir}")
script = CommandScript(f"{vlsi_dir}/run-sim.sh")
for w in my_workloads:
    # run simulation with FSDB + commit log dumping (this make target defined in custom.mk)
    binary_path = f"{gemmini_dir}/build/bareMetalC/{w}-baremetal"
    outfile = f"{root_dir}/data/vcs_output/{w}-baremetal.log"
    if overwrite or not os.path.exists(outfile):
        script.f.write(f"{bsub} make sim-rtl-debug-out CONFIG={my_rtl} BINARY={binary_path} OUTFILE={outfile}\n".lstrip())
        script.f.write(f"sleep 1\n")
script.close()
# RUN THIS

Make sure chipyard environment sourced
cd /tools/scratch/vinaya/power-prediction/power-mappings-chipyard/vlsi
./run-sim.sh


### Joules Power

In [8]:
print("Make sure chipyard environment sourced")
print(f"cd {vlsi_dir}")
binary_path = f"{gemmini_dir}/build/bareMetalC/{micro_benchmarks[0]}-baremetal"

print(f"{bsub} make power-rtl CONFIG={my_rtl} BINARY={binary_path}\n")
# RUN ONCE - Joules will synthesize the design 
#   (it's ok if this step errors during the report_power step - we generate reports below)

Make sure chipyard environment sourced
cd /tools/scratch/vinaya/power-prediction/power-mappings-chipyard/vlsi
bsub -q rhel7 make power-rtl CONFIG=CustomGemminiSoCConfig BINARY=/tools/scratch/vinaya/power-prediction/power-mappings-chipyard/generators/gemmini/software/gemmini-rocc-tests/build/bareMetalC/mvin_cache_hit_microbenchmark_random-baremetal



In [20]:
import yaml

print("Make sure chipyard environment sourced")
print(f"cd {vlsi_dir}")
script = CommandScript(f"{vlsi_dir}/run-power.sh")

for w in my_workloads:
    waveform_path = f"{vlsi_dir}/output/chipyard.harness.TestHarness.CustomGemminiSoCConfig/{w}-baremetal.fsdb"

    # initial joules power config
    report_name = f"{root_dir}/data/joules_output/{w}-baremetal"
    report_fpath = f"{root_dir}/data/joules_output/{w}-baremetal-gemmini.profile.png.data"
    if not overwrite and os.path.exists(report_fpath): continue
    cfg = {
        'power.inputs.saifs': [],
        'power.inputs.waveforms': [],
        'power.joules.version': '221',
        'vlsi.core.power_tool': 'joules',
        'power.inputs.report_configs': [
            {'waveform_path': waveform_path,
            'toggle_signal': '/ChipTop/clock_uncore_clock',
            'inst': "/ChipTop/system/tile_prci_domain/tile_reset_domain_tile/gemmini",
            'num_toggles': 100,
            'report_name': report_fpath.replace('.profile.png.data',''),
            'output_formats': ['report', 'plot_profile']},
            {'waveform_path': waveform_path,
            'toggle_signal': '/ChipTop/clock_uncore_clock',
            'inst': "/ChipTop/system/tile_prci_domain/tile_reset_domain_tile/gemmini/spad/acc_mems_0",
            'num_toggles': 100,
            'report_name': f"{report_name}-acc_mems_0",
            'output_formats': ['plot_profile']},
            {'waveform_path': waveform_path,
            'toggle_signal': '/ChipTop/clock_uncore_clock',
            'inst': "/ChipTop/system/tile_prci_domain/tile_reset_domain_tile/gemmini/spad/acc_mems_1",
            'num_toggles': 100,
            'report_name': f"{report_name}-acc_mems_1",
            'output_formats': ['plot_profile']},
            {'waveform_path': waveform_path,
            'toggle_signal': '/ChipTop/clock_uncore_clock',
            'inst': "/ChipTop/system/tile_prci_domain/tile_reset_domain_tile/gemmini/spad/spad_mems_0",
            'num_toggles': 100,
            'report_name': f"{report_name}-spad_mems_0",
            'output_formats': ['plot_profile']},
            {'waveform_path': waveform_path,
            'toggle_signal': '/ChipTop/clock_uncore_clock',
            'inst': "/ChipTop/system/tile_prci_domain/tile_reset_domain_tile/gemmini/spad/spad_mems_1",
            'num_toggles': 100,
            'report_name': f"{report_name}-spad_mems_1",
            'output_formats': ['plot_profile']},
            {'waveform_path': waveform_path,
            'toggle_signal': '/ChipTop/clock_uncore_clock',
            'inst': "/ChipTop/system/tile_prci_domain/tile_reset_domain_tile/gemmini/spad/spad_mems_2",
            'num_toggles': 100,
            'report_name': f"{report_name}-spad_mems_2",
            'output_formats': ['plot_profile']},
            {'waveform_path': waveform_path,
            'toggle_signal': '/ChipTop/clock_uncore_clock',
            'inst': "/ChipTop/system/tile_prci_domain/tile_reset_domain_tile/gemmini/spad/spad_mems_3",
            'num_toggles': 100,
            'report_name': f"{report_name}-spad_mems_3",
            'output_formats': ['plot_profile']},
            {'waveform_path': waveform_path,
            'toggle_signal': '/ChipTop/clock_uncore_clock',
            'inst': "/ChipTop/system/tile_prci_domain/tile_reset_domain_tile/gemmini/ex_controller/mesh",
            'num_toggles': 100,
            'report_name': f"{report_name}-mesh",
            'output_formats': ['plot_profile']}]
    }

    yaml_cfg_fpath = f"{root_dir}/flows/yaml_configs/{w}-baremetal.yml"
    with open(yaml_cfg_fpath,'w') as yf:
        yaml.dump(cfg, yf, sort_keys=False)

    make_target = f"""redo-power-rtl args="--only_step report_power" """
    binary_path = f"{gemmini_dir}/build/bareMetalC/{w}-baremetal"

    script.f.write(f"{bsub} make {make_target} extra={yaml_cfg_fpath} CONFIG={my_rtl} BINARY={binary_path}\n")
    # need this or jobs sometimes all crash from overlapping (I think)
    script.f.write(f"sleep 10\n")

script.close()
# RUN THIS

Make sure chipyard environment sourced
cd /tools/scratch/vinaya/power-prediction/power-mappings-chipyard/vlsi
./run-power.sh


In [10]:
import numpy as np



In [19]:
arr = np.array([1, 2, 3, 4, 5])
arr2 = np.full_like(arr, 50)
(arr*arr2).sum()* (10 ** -2)

7.5